# भारत Avatar Platform — SadTalker Video Generation Pipeline

**India Innovates 2026 — Google Colab Setup**

Run all 3 cells in order. Cell 1 takes ~5 min.

**Pipeline:** Admin message → Edge TTS (.mp3) → SadTalker (.mp4) → Broadcast

**Set runtime to GPU (T4) before running!**

In [ ]:
# ============================================================
# CELL 1 — Install SadTalker + Dependencies + Patches
# ============================================================
import os

if not os.path.exists('/content/SadTalker'):
    !git clone https://github.com/cedro3/SadTalker.git &> /dev/null
    print('✅ SadTalker cloned')
else:
    print('✅ SadTalker already exists')

%cd /content/SadTalker

!pip install torch torchvision torchaudio --quiet
!apt update -qq && apt install ffmpeg -y &> /dev/null
print('✅ ffmpeg installed')

!pip install -r requirements.txt --quiet

print('Downloading pre-trained models...')
!rm -rf checkpoints
!bash scripts/download_models.sh

!pip install kornia facexlib gfpgan yacs tqdm basicsr edge-tts flask flask-cors cloudflared --quiet

# ---- NumPy 2.0 Patches (proven working on Colab Python 3.12) ----
print('\n⏳ Applying patches for NumPy 2.0 compatibility...')
!sed -i 's/np.VisibleDeprecationWarning/DeprecationWarning/g' /content/SadTalker/src/face3d/util/preprocess.py
!sed -i 's/functional_tensor/functional/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py 2>/dev/null || true
!sed -i 's/np.float/float/g' /usr/local/lib/python3.12/dist-packages/facexlib/alignment/awing_arch.py
!sed -i 's/np.array(\[w0, h0, s, t\[0\], t\[1\]\])/np.array([float(w0), float(h0), float(s), float(t[0]), float(t[1])])/' /content/SadTalker/src/face3d/util/preprocess.py
print('✅ Environment Patched.')


# --- FACTORY RESET & FIX FFMPEG DIMENSIONS ---
print("⚙️ Resetting and applying FFmpeg dimension fix...")
!git checkout src/facerender/animate.py 2>/dev/null
path = "src/facerender/animate.py"
with open(path, 'r') as f: content = f.read()
content = content.replace('imageio.mimsave(path, result,  fps=float(25))', 'imageio.mimsave(path, result,  fps=float(25), macro_block_size=2)')
content = content.replace('imageio.mimsave(path, result, fps=float(25))', 'imageio.mimsave(path, result, fps=float(25), macro_block_size=2)')
with open(path, 'w') as f: f.write(content)
print("✅ Macro-block patch applied (set to 2).")

%cd /content
print('\n✅ Cell 1 Complete!')

In [ ]:
# ============================================================
# CELL 2 — Helper Functions + Flask API + Cloudflare Tunnel
# ============================================================
import os
import uuid
import glob
import shutil
import subprocess
import asyncio
import re
import time
import threading
import requests
import base64
import edge_tts
from flask import Flask, request, send_file, jsonify
from flask_cors import CORS
from PIL import Image

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  🔑  BHASHINI CREDENTIALS & CONFIG
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BHASHINI_UDYAT_KEY     = "37e590f73c-b1d0-42b5-ad5b-233643692975"
BHASHINI_INFERENCE_KEY = "9FnD4mGS91aFLPW7RqW1iNlDQLgsAOn8BooeNCtpeg5O6JU7-I8tJi0OfxbcKIQo"
BHASHINI_USER_ID       = "a5fc8fbb04804d1bb76793f007196d80"
BHASHINI_INFERENCE_URL = "https://dhruva-api.bhashini.gov.in/services/inference/pipeline"

SVC_INDO_ARYAN = "ai4bharat/indic-tts-coqui-indo_aryan-gpu--t4"
SVC_DRAVIDIAN  = "ai4bharat/indic-tts-coqui-dravidian-gpu--t4"
SVC_MISC       = "ai4bharat/indic-tts-coqui-misc-gpu--t4"

VOICE_CONFIG = {
    ("hi", "male"): {
        "language": "hi", "gender": "male", "service_id": SVC_INDO_ARYAN,
        "edge_voice": "hi-IN-MadhurNeural", "edge_rate": "-10%", "edge_pitch": "-3Hz"
    },
    ("mr", "female"): {
        "language": "mr", "gender": "female", "service_id": SVC_INDO_ARYAN,
        "edge_voice": "mr-IN-AarohiNeural", "edge_rate": "-5%", "edge_pitch": "+2Hz"
    },
    ("ta", "male"): {
        "language": "ta", "gender": "male", "service_id": SVC_DRAVIDIAN,
        "edge_voice": "ta-IN-ValluvarNeural", "edge_rate": "-8%", "edge_pitch": "-2Hz"
    },
    ("bn", "female"): {
        "language": "bn", "gender": "female", "service_id": SVC_INDO_ARYAN,
        "edge_voice": "bn-IN-TanishaaNeural", "edge_rate": "-5%", "edge_pitch": "+1Hz"
    },
    ("en", "male"): {
        "language": "en", "gender": "male", "service_id": SVC_MISC,
        "edge_voice": "en-IN-PrabhatNeural", "edge_rate": "0%", "edge_pitch": "0Hz"
    },
    ("hi", "female"): {
        "language": "hi", "gender": "female", "service_id": SVC_INDO_ARYAN,
        "edge_voice": "hi-IN-SwaraNeural", "edge_rate": "-10%", "edge_pitch": "-3Hz"
    },
    ("mr", "male"): {
        "language": "mr", "gender": "male", "service_id": SVC_INDO_ARYAN,
        "edge_voice": "mr-IN-ManoharNeural", "edge_rate": "-5%", "edge_pitch": "-3Hz"
    },
    ("bn", "male"): {
        "language": "bn", "gender": "male", "service_id": SVC_INDO_ARYAN,
        "edge_voice": "bn-IN-BashkarNeural", "edge_rate": "-5%", "edge_pitch": "-3Hz"
    }
}

def _bhashini_tts(text, cfg, out_path):
    payload = {
        "pipelineTasks": [{"taskType": "tts", "config": {
            "serviceId": cfg["service_id"], "language": {"sourceLanguage": cfg["language"]},
            "gender": cfg["gender"], "samplingRate": 22050,
        }}],
        "inputData": {"input": [{"source": text}], "audio": [{"audioContent": None}]}
    }
    headers = {
        "Authorization": BHASHINI_INFERENCE_KEY, "userID": BHASHINI_USER_ID,
        "ulcaApiKey": BHASHINI_UDYAT_KEY, "Content-Type": "application/json",
    }
    try:
        resp = requests.post(BHASHINI_INFERENCE_URL, json=payload, headers=headers, timeout=30)
        resp.raise_for_status()
        audio_b64 = resp.json()["pipelineResponse"][0]["audio"][0]["audioContent"]
        with open(out_path, "wb") as fh: fh.write(base64.b64decode(audio_b64))
        return True
    except Exception as e:
        print(f"  ❌ Bhashini error: {e}")
        return False

async def _edge_tts_async(text, cfg, out_path):
    comm = edge_tts.Communicate(text=text, voice=cfg["edge_voice"], rate=cfg.get("edge_rate", "0%"), pitch=cfg.get("edge_pitch", "0Hz"))
    await comm.save(out_path)

def generate_voice_sync(text, lang, gender, output_path):
    cfg = VOICE_CONFIG.get((lang, gender))
    if not cfg:
        print(f"⚠️ Warning: No config found for {lang}-{gender}. Using default Edge-TTS hi-male fallback")
        cfg = {
            "language": lang, "gender": gender, "service_id": SVC_INDO_ARYAN,
            "edge_voice": "hi-IN-MadhurNeural", "edge_rate": "-10%", "edge_pitch": "-3Hz"
        }
        
    print(f"🎙 Generating voice for {lang}-{gender}...")
    if not _bhashini_tts(text, cfg, output_path):
        print(f"  🔄 Falling back to Edge TTS ({cfg['edge_voice']})")
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(_edge_tts_async(text, cfg, output_path))
        loop.close()
        
    print(f'  ✅ Audio: {output_path} ({os.path.getsize(output_path)} bytes)')
    return output_path

def generate_sadtalker_video(photo_path, audio_path, output_path):
    if not os.path.exists(photo_path):
        raise RuntimeError(f'Photo not found: {photo_path}')
    if not os.path.exists(audio_path):
        raise RuntimeError(f'Audio not found: {audio_path}')
        
    # --- IMAGE RESIZING PATCH ---
    try:
        img = Image.open(photo_path)
        new_w = int((720 * (img.width / img.height)) // 16) * 16
        img.resize((new_w, 720), Image.Resampling.LANCZOS).save(photo_path)
        print(f"🎨 Image optimized to {new_w}x720")
    except Exception as e:
        print(f"⚠️ Image resize failed: {e}")

    print(f'  🎥 SadTalker: photo={photo_path} audio={audio_path}')
    result_dir = '/content/results'
    os.makedirs(result_dir, exist_ok=True)
    cmd = [
        'python', 'inference.py',
        '--driven_audio', audio_path,
        '--source_image', photo_path,
        '--result_dir', result_dir,
        '--still',
        '--preprocess', 'full',
        '--enhancer', 'gfpgan'
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, cwd='/content/SadTalker')
    if result.returncode != 0:
        print(f'  STDERR: {result.stderr[-500:]}')
        raise RuntimeError(f'SadTalker failed: {result.stderr[-300:]}')
    files = glob.glob(f'{result_dir}/*/*.mp4') or glob.glob(f'{result_dir}/*.mp4')
    if not files:
        raise RuntimeError('No mp4 found after generation')
    latest = max(files, key=os.path.getctime)
    shutil.copy2(latest, output_path)
    print(f'  ✅ Video: {output_path} ({os.path.getsize(output_path)/(1024*1024):.1f} MB)')
    return output_path

print('✅ Helper functions ready')

# ---- Flask API ----
flask_app = Flask(__name__)
CORS(flask_app)
os.makedirs('/content/uploads', exist_ok=True)
os.makedirs('/content/server_results', exist_ok=True)

@flask_app.route('/generate', methods=['POST'])
def api_generate():
    text = request.form.get('text', '')
    lang = request.form.get('lang', 'hi')
    gender = request.form.get('gender', 'male')
    if not text:
        return jsonify({'error': 'Text is required'}), 400
    job_id = str(uuid.uuid4())[:8]
    print(f'\n🚀 Job {job_id}: lang={lang} gender={gender}')
    photo_path = '/content/default_avatar.jpg'
    if 'photo' in request.files:
        photo = request.files['photo']
        if photo.filename:
            photo_path = f'/content/uploads/{job_id}_photo.png'
            photo.save(photo_path)
    if not os.path.exists(photo_path):
        return jsonify({'error': 'No photo available'}), 400
    try:
        audio_path = f'/content/server_results/{job_id}_audio.mp3'
        generate_voice_sync(text, lang, gender, audio_path)
        video_path = f'/content/server_results/{job_id}_video.mp4'
        generate_sadtalker_video(photo_path, audio_path, video_path)
        return send_file(video_path, mimetype='video/mp4', as_attachment=True, download_name=f'avatar_{job_id}.mp4')
    except Exception as e:
        import traceback; traceback.print_exc()
        return jsonify({'error': str(e)}), 500

@flask_app.route('/health', methods=['GET'])
def api_health():
    import torch
    return jsonify({'status': 'ok', 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'})

threading.Thread(target=lambda: flask_app.run(host='0.0.0.0', port=5000, threaded=True), daemon=True).start()
print('✅ Flask on port 5000')
time.sleep(2)

# ---- Cloudflare Tunnel ----
print('\n🌐 Starting tunnel...')
tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:5000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(10)
tunnel_url = None
for _ in range(30):
    line = tunnel.stderr.readline().decode('utf-8', errors='ignore')
    if '.trycloudflare.com' in line:
        urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', line)
        if urls:
            tunnel_url = urls[0]
            break
    time.sleep(1)

if not tunnel_url:
    remaining = tunnel.stderr.read(4096).decode('utf-8', errors='ignore')
    urls = re.findall(r'https://[\w-]+\.trycloudflare\.com', remaining)
    if urls: tunnel_url = urls[0]

if tunnel_url:
    print(f'\n{"="*60}')
    print(f'🚀 COLAB_URL={tunnel_url}')
    print(f'   Health: {tunnel_url}/health')
    print(f'{"="*60}')
else:
    print('❌ Could not get tunnel URL')

print('\n⏳ Keep this cell running!')
while True:
    time.sleep(60)
